In [5]:
import pandas as pd

# This loads the data and shows the first 5 rows
df = pd.read_csv(r"C:\Users\Placid\Downloads\archive (2)\credit_risk_dataset.csv")
display(df.head())

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [6]:
# 1. Check for missing values
print("Missing values before cleaning:")
print(df.isnull().sum())

# 2. Clean the missing data 
# We use the median rather than the average (mean) because financial data 
# usually contains extreme outliers that skew averages.
df['person_emp_length'] = df['person_emp_length'].fillna(df['person_emp_length'].median())
df['loan_int_rate'] = df['loan_int_rate'].fillna(df['loan_int_rate'].median())

# 3. Save the clean dataset for our SQL database
df.to_csv('../data/clean_credit_data.csv', index=False)
print("\nMissing values fixed and clean_credit_data.csv saved!")

Missing values before cleaning:
person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

Missing values fixed and clean_credit_data.csv saved!


In [7]:
import sqlite3

# 1. Create a local SQL database inside your data folder
conn = sqlite3.connect('../data/credit_risk.db')

# 2. Upload our clean Pandas dataframe directly into a SQL table named 'credit_data'
df.to_sql('credit_data', conn, if_exists='replace', index=False)

# 3. Write your first analytical SQL Query! 
# Let's segment the financial risk to see which loan grades default the most.
query = """
SELECT 
    loan_grade,
    COUNT(client_id) as total_loans,
    AVG(loan_status) * 100 as default_rate_percentage
FROM credit_data
GROUP BY loan_grade
ORDER BY loan_grade ASC;
"""

# 4. Execute the SQL query and display the results
sql_results = pd.read_sql_query(query, conn)
display(sql_results)

DatabaseError: Execution failed on sql '
SELECT 
    loan_grade,
    COUNT(client_id) as total_loans,
    AVG(loan_status) * 100 as default_rate_percentage
FROM credit_data
GROUP BY loan_grade
ORDER BY loan_grade ASC;
': no such column: client_id